<a href="https://colab.research.google.com/github/deepanathanrajendiran-hub/sft-code-review/blob/feat%2Fcorpo/corpo_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 0 — Install dependencies (Unsloth canonical Colab branch) + clone feat/corpo
#
# Mirrors the install logic from Unsloth's Qwen2.5_(3B)-GRPO notebook, cells 4+5:
#   https://raw.githubusercontent.com/unslothai/notebooks/main/nb/Qwen2.5_(3B)-GRPO.ipynb
#
# Why this exact shape (do not "simplify"):
#   - UNSLOTH_VLLM_STANDBY=1   → +30% context-length headroom (unsloth-specific)
#   - upgrade `uv` first       → uv's resolver is stricter than pip's; avoids the
#                                pip-picks-wrong-trl mistakes we hit on Path B
#   - GPU-aware vllm/triton    → T4 needs vllm==0.9.2 + triton==3.2.0; A100/L4/H100
#                                use vllm==0.15.1 + latest triton. Latest vllm
#                                doesn't work on T4 (silent crash on import).
#   - ONE uv-call bundle       → vllm + numpy + pil + torchvision + bitsandbytes +
#                                xformers + unsloth resolved together so versions
#                                stay consistent (no pip-installs-X-then-uv-overrides-Y).
#   - trl==0.22.2 --no-deps    → avoids TRL 0.24's vllm_ascend + mergekit imports.
#                                --no-deps so trl doesn't drag in transitive packages
#                                that would fight the unsloth-bundled versions.
#   - transformers==4.56.2     → matches what unsloth's wheel was built against.
#   - peft==0.17.1 --no-deps   → 0.18+ added _maybe_shard_state_dict_for_tp which
#                                imports transformers.integrations.tensor_parallel.
#                                EmbeddingParallel — that symbol doesn't exist
#                                until transformers 4.57+. Pinning to 0.17.1 (last
#                                pre-TP release, 2025-08-21) avoids the conflict.
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"

!pip install --upgrade -qqq uv

if "COLAB_" not in "".join(os.environ.keys()):
    # Non-Colab environment (local dev, RunPod, etc.): unsloth's simple path
    !pip install unsloth vllm
else:
    # Resolve currently-installed numpy + pillow versions to avoid churning them
    try:
        import numpy, PIL
        _numpy = f"numpy=={numpy.__version__}"
        _pil   = f"pillow=={PIL.__version__}"
    except Exception:
        _numpy, _pil = "numpy", "pillow"

    # GPU-aware vllm + triton pinning
    try:
        import subprocess
        is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except Exception:
        is_t4 = False
    _vllm, _triton = ("vllm==0.9.2", "triton==3.2.0") if is_t4 else ("vllm==0.15.1", "triton")

    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
    !uv pip install -qqq --no-deps --upgrade "torchao>=0.16.0"

!uv pip install -qqq transformers==4.56.2
!uv pip install -qqq --no-deps trl==0.22.2
!uv pip install -qqq --no-deps peft==0.17.1

# Project-specific extras (not part of the canonical Unsloth recipe):
#   openai — DeepSeek V4-Pro judge HTTP client
!uv pip install -qqq "openai>=1.0.0"

# Clone the project at feat/corpo (private repo)
from google.colab import userdata
GITHUB_PAT = userdata.get('GITHUB_PAT')
!rm -rf /content/sft
!git clone --branch feat/corpo --depth 1 \
    https://{GITHUB_PAT}@github.com/deepanathanrajendiran-hub/sft-code-review.git \
    /content/sft
!cp /content/sft/*.py /content/sft/pyproject.toml /content/
!cp -r /content/sft/tests /content/
os.chdir("/content")

!ls /content/corpo_*.py /content/swecare_*.py /content/ood_metrics.py
print("\nVersion check:")
import trl, vllm, torch, datasets, peft, transformers
print(f"  trl          : {trl.__version__}     (expected 0.22.2)")
print(f"  transformers : {transformers.__version__}    (expected 4.56.2)")
print(f"  vllm         : {vllm.__version__}     (T4=0.9.2, else=0.15.1)")
print(f"  torch        : {torch.__version__}")
print(f"  datasets     : {datasets.__version__}")
print(f"  peft         : {peft.__version__}    (expected 0.17.1)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 99.0 MB/s eta 0:00:00
Cloning into '/content/sft'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 48 (delta 0), reused 22 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (48/48), 177.48 KiB | 16.13 MiB/s, done.
/content/corpo_decision_gate.py  /content/ood_metrics.py
/content/corpo_reward.py	 /content/swecare_loader.py
/content/corpo_trainer.py	 /content/swecare_split.py
/content/corpo_train.py

Version check:


/usr/local/lib/python3.12/dist-packages/torchao/quantization/quant_api.py:1745: SyntaxWarning: invalid escape sequence '\.'
  * regex for parameter names, must start with `re:`, e.g. `re:language\.layers\..+\.q_proj.weight`.


  trl          : 0.22.2     (expected 0.22.2)
  transformers : 4.56.2    (expected 4.56.2)
  vllm         : 0.15.1     (T4=0.9.2, else=0.15.1)
  torch        : 2.9.1+cu128
  datasets     : 4.3.0
  peft         : 0.17.1    (expected 0.17.1)


In [2]:
# Cell 1 — Mount Drive, load secrets, verify v4 backup
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
os.environ['DEEPSEEK_API_KEY'] = userdata.get('DEEPSEEK_API_KEY')
os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_DEFAULT_REGION'] = 'us-west-2'

# v4 adapter paths — USER MUST ensure backup exists before this cell runs
V4_ADAPTER = '/content/drive/MyDrive/sft/code-reviewer-lora-v4-traces'
V4_BACKUP  = '/content/drive/MyDrive/sft/code-reviewer-lora-v4-traces-backup'
assert os.path.exists(V4_BACKUP + '/adapter_config.json'), \
    f"v4 backup missing! Create one BEFORE running: cp -r {V4_ADAPTER} {V4_BACKUP}"
print(f"v4 adapter:  {V4_ADAPTER}")
print(f"v4 backup:   {V4_BACKUP}")

Mounted at /content/drive
v4 adapter:  /content/drive/MyDrive/sft/code-reviewer-lora-v4-traces
v4 backup:   /content/drive/MyDrive/sft/code-reviewer-lora-v4-traces-backup


In [3]:
# Cell 2 — Build train/eval splits, extract CLEAN defect labels, and score v4 (THE GATE)
# v5: no base-sample cache / no opponent. We extract clean, grounded defect tuples from the
# human PR threads (label_defects.py) and measure v4's JUDGE-INDEPENDENT recall + hallucination.
# Requires DEEPSEEK_API_KEY (Cell 1). No GPU needed for this cell.
import json, os, random

# --- training prompts from dev split; eval from test split (disjoint) ---
!python /content/swecare_loader.py --split dev \
    --output /content/ood_dev_prompts_raw.jsonl \
    --train-jsonl /content/drive/MyDrive/sft/train_dataset_clean.jsonl
with open('/content/ood_dev_prompts_raw.jsonl') as f:
    dev_rows = [json.loads(l) for l in f if l.strip()]
sample = random.Random(42).sample(dev_rows, min(1500, len(dev_rows)))
with open('/content/ood_train_prompts.jsonl', 'w') as f:
    for r in sample: f.write(json.dumps(r) + '\n')
!python /content/swecare_loader.py --split test \
    --output /content/ood_input.jsonl \
    --train-jsonl /content/drive/MyDrive/sft/train_dataset_clean.jsonl
print(f"dev pool {len(dev_rows)}   train sample {len(sample)}")

# --- Stage 1: extract clean defect tuples (drops questions/replies/style nits; grounds to diff) ---
os.makedirs('/content/cache', exist_ok=True)
!python /content/label_defects.py --input /content/ood_train_prompts.jsonl --output /content/cache/defect_labels_train.jsonl
!python /content/label_defects.py --input /content/ood_input.jsonl        --output /content/cache/defect_labels_eval.jsonl

# --- THE GATE: v4 (and base) recall + hallucination on the clean EVAL labels ---
# Uses the existing ood_preds_v4.jsonl (already has v4_pred + base_pred).
!python /content/score_v5.py \
    --preds /content/drive/MyDrive/sft/ood_preds_v4.jsonl \
    --labels /content/cache/defect_labels_eval.jsonl \
    --pred-fields v4_pred base_pred

print("\n[GATE] Decision:")
print("  - v4 recall well below 1.0  -> headroom exists, RL is viable. Record v4 recall + halluc; proceed to Cell 3.")
print("  - v4 recall already ~saturated -> RL can only restore, not exceed. STOP and pivot to data (v4.1).")
print("  - This v4 recall/halluc pair IS the bar Cell 5/Cell 7 must beat (recall UP, halluc <= v4).")

[swecare_loader] excluding train repos: ['fastapi', 'pydantic', 'scikit-learn', 'transformers']





[swecare_loader] split=dev kept 6648 / 7086 rows (438 excluded by repo-overlap)
[swecare_loader] wrote 6648 rows to /content/ood_dev_prompts_raw.jsonl
[swecare_loader] excluding train repos: ['fastapi', 'pydantic', 'scikit-learn', 'transformers']
[swecare_loader] split=test kept 632 / 671 rows (39 excluded by repo-overlap)
[swecare_loader] wrote 632 rows to /content/ood_input.jsonl
dev pool 6648   train sample 1500
[label_defects] extracting defects from 1500 records
[label_defects] wrote 1500 records: 892 defect tuples, 818 verified-clean diffs, 12 ambiguous (no grounded defects but ungrounded defect comments — excluded from training by load_defect_labels) -> /content/cache/defect_labels_train.jsonl
[label_defects] extracting defects from 632 records
[label_defects] wrote 632 records: 373 defect tuples, 344 verified-clean diffs, 7 ambiguous (no grounded defects but ungrounded defect co

In [4]:
!cp -r /content/cache /content/drive/MyDrive/sft/cache-v5.2

In [5]:
# Cell 3 — Pre-training variance gate + auto-pick R_min for v5 (~10-15 min, ~$1 DeepSeek)
#
# v5 scores v4 rollouts with the VERIFIABLE reward (F1 on labeled + claim-penalty on clean +
# grounding + length) — no opponent, no quality judge. The gate confirms the reward has
# within-group spread (>=0.10) so advantages don't collapse, and prints p25/p33/p40/p50
# R_min candidates (CoRPO correctness boundary). This cell auto-extracts the p33 default.
import subprocess, re, sys

print("[cell3] running v5 variance gate (verifiable reward against clean defect labels)...")
result = subprocess.run(
    ["python", "/content/corpo_train.py", "--variance-gate-only",
     "--v4-adapter", V4_ADAPTER,
     "--v4-backup",  V4_BACKUP,
     "--train-prompts", "/content/ood_train_prompts.jsonl",
     "--defect-labels", "/content/cache/defect_labels_train.jsonl",
     "--output-dir", "/content/corpo-out"],
    capture_output=True, text=True,
)
print(result.stdout)
print(result.stderr, file=sys.stderr)

# Distinguish a real gate FAIL (flat reward) from a crash (code/env error). The gate only
# prints "verdict: PASS/FAIL" if it ran to completion; a traceback means it crashed.
reached_verdict = ("[variance-gate] verdict:" in result.stderr)
if result.returncode != 0:
    if not reached_verdict:
        raise RuntimeError(
            "[variance-gate] CRASHED before a verdict (see traceback above) — a code/env error, "
            "NOT a flat-reward failure. Fix the error (e.g. git pull + re-copy /content/*.py) and re-run."
        )
    raise RuntimeError(
        "[variance-gate] FAIL — within-group reward std < 0.10 (flat reward). Do NOT train. "
        "Inspect the histogram above; the fix is to rebalance the reward, not to train. Paste the output."
    )

m = re.search(r"recommended default: p33 = ([\d.]+)", result.stderr)
if not m:
    raise RuntimeError("[variance-gate] PASSED but couldn't parse R_min; set R_MIN manually from the printout.")
R_MIN = float(m.group(1))
print(f"\n[cell3] PASS — auto-selected R_MIN = {R_MIN}  (p33 of the v4 verifiable-reward distribution)")
print(f"[cell3] If the histogram looks bimodal, set R_MIN at the trough manually, then run Cell 4.")

[cell3] running v5 variance gate (verifiable reward against clean defect labels)...
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 06-10 23:12:32 [utils.py:261] non-default args: {'max_model_len': 8192, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'enable_lora': True, 'max_lora_rank': 64, 'model': 'unsloth/Qwen2.5-Coder-7B-Instruct'}
INFO 06-10 23:12:50 [model.py:541] Resolved architecture: Qwen2ForCausalLM
INFO 06-10 23:12:50 [model.py:1561] Using max model len 8192
INFO 06-10 23:12:50 [scheduler.py:226] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 06-10 23:12:50 [vllm.py:624] Asynchronous scheduling is enabled.
WARNING 06-10 23:12:51 [system_utils.py:140] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more inf

[corpo_train] verified v4 backup at /content/drive/MyDrive/sft/code-reviewer-lora-v4-traces-backup
[corpo_train] pre-flight: TRL exposes loss_type ✓
[corpo_train] pre-flight: max_prompt_length is configurable (set to 6144 in training) ✓
[corpo_train] pre-flight: batch ok (pdtbs=num_generations=8, grad_accum=prompts_per_step=4) ✓
[corpo_train] skipped 12 ambiguous label records (no grounded defects but ungrounded defect comments exist)
[variance-gate] dropped 12/1500 prompts with ambiguous/missing defect labels (matches training filter)
[variance-gate] generating 50 x 8 v4 rollouts
(EngineCore_DP0 pid=5367) 
Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
(EngineCore_DP0 pid=5367) 
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:01<00:04,  1.39s/it]
(EngineCore_DP0 pid=5367) 
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:01<00:01,  1.23it/s]
(EngineCore_DP0 pid=5367) 
Loading safetensors checkpoint shards:  75% Completed | 3

In [7]:
  # Diagnose + repair merged v4 (shard-integrity verified)
  import os, glob, json, shutil, gc

  def shards_ok(d):
      idx = os.path.join(d, 'model.safetensors.index.json')
      if not os.path.exists(idx): return False
      total = json.load(open(idx)).get('metadata', {}).get('total_size', 0)
      have = sum(os.path.getsize(p) for p in glob.glob(os.path.join(d, '*.safetensors')))
      print(f"{d}: index expects {total/1e9:.2f} GB, shards on disk {have/1e9:.2f} GB")
      return total > 0 and have >= total

  DRIVE_MERGED = '/content/drive/MyDrive/sft/sft-v4-merged-for-eval'
  V4_MERGED    = '/content/sft-v4-merged'

  print("Drive source ok:", shards_ok(DRIVE_MERGED))
  print("Local copy ok:  ", shards_ok(V4_MERGED))

  shutil.rmtree(V4_MERGED, ignore_errors=True)
  if shards_ok(DRIVE_MERGED):
      print("re-copying from Drive...")
      os.system(f"cp -r {DRIVE_MERGED} {V4_MERGED}")
  if not shards_ok(V4_MERGED):
      print("rebuilding merged v4 from the adapter (~10 min, CPU + base download)...")
      shutil.rmtree(V4_MERGED, ignore_errors=True)
      import torch
      from transformers import AutoModelForCausalLM, AutoTokenizer
      from peft import PeftModel
      _b = AutoModelForCausalLM.from_pretrained('unsloth/Qwen2.5-Coder-7B-Instruct', dtype=torch.bfloat16)
      _m = PeftModel.from_pretrained(_b, V4_ADAPTER).merge_and_unload()
      _m.save_pretrained(V4_MERGED, safe_serialization=True)   # local SSD = atomic
      AutoTokenizer.from_pretrained(V4_ADAPTER).save_pretrained(V4_MERGED)
      del _m, _b; gc.collect()
  assert shards_ok(V4_MERGED), "merged v4 still failing shard check"
  print("✓ merged v4 verified at", V4_MERGED)


/content/drive/MyDrive/sft/sft-v4-merged-for-eval: index expects 15.23 GB, shards on disk 1.09 GB
Drive source ok: False
/content/sft-v4-merged: index expects 15.23 GB, shards on disk 1.09 GB
Local copy ok:   False
/content/drive/MyDrive/sft/sft-v4-merged-for-eval: index expects 15.23 GB, shards on disk 1.09 GB
rebuilding merged v4 from the adapter (~10 min, CPU + base download)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/config.py:165: UserWarning: Unexpected keyword arguments ['alora_invocation_tokens', 'arrow_config', 'ensure_weight_tying', 'lora_ga_config', 'peft_version', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


AttributeError: 'list' object has no attribute 'keys'

In [8]:
  import os, json, gc
  from transformers import AutoTokenizer

  base_tok = AutoTokenizer.from_pretrained('unsloth/Qwen2.5-Coder-7B-Instruct')

  def dir_chat_template(d):  # read template straight from files, no tokenizer instantiation
      p = os.path.join(d, 'chat_template.jinja')
      if os.path.exists(p): return open(p).read()
      return json.load(open(os.path.join(d, 'tokenizer_config.json'))).get('chat_template')

  tpl = dir_chat_template(V4_ADAPTER)
  assert tpl is None or tpl == base_tok.chat_template, "adapter chat_template differs from base!"

  base_tok.save_pretrained(V4_MERGED)
  try: del _m, _b
  except NameError: pass
  gc.collect()
  assert shards_ok(V4_MERGED), "merged v4 still failing shard check"
  print("✓ merged v4 complete at", V4_MERGED)

/content/sft-v4-merged: index expects 15.23 GB, shards on disk 15.23 GB
✓ merged v4 complete at /content/sft-v4-merged


In [10]:
!cd /content/sft && git pull --ff-only && cp /content/sft/*.py /content/


remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 12 (delta 8), reused 8 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (12/12), 5.50 KiB | 937.00 KiB/s, done.
From https://github.com/deepanathanrajendiran-hub/sft-code-review
   9016e8b..9bcb628  feat/corpo -> origin/feat/corpo
Updating 9016e8b..9bcb628
Fast-forward
 corpo_train.ipynb | 118 ++++++++++++++++++++++++++++++++++++++++++++----------
 corpo_train.py    |  73 +++++++++++++++++++++++++++++++--
 2 files changed, 166 insertions(+), 25 deletions(-)


In [11]:
# Cell 4 — Train v5.2 (verifiable-reward CoRPO) — fixes the v5.0/v5.1 pipeline defects
#
# Post-mortem of Runs #1-3 and v5.0/v5.1 found that NO earlier RL run tested the designed
# reward. v5.2 fixes, in order of impact:
#   1. max_prompt_length=6144 (corpo_train.py). TRL's GRPOConfig DEFAULTS to 512 and
#      left-truncates every prompt — ~87% of training prompts lost the system message
#      and most of the diff in every earlier run. The policy was scored on defects it
#      could not see.
#   2. --v4-merged: policy = merged-v4 weights + FRESH LoRA. TRL's KL reference for PEFT
#      models is "adapters disabled" — with the old adapter-loading that meant BASE
#      (pulling the policy away from v4); now it is actually v4.
#   3. Truncated rollouts (unclosed <think>) score 0 instead of leaking reasoning text
#      through the extractor into the reward.
#   4. Ambiguous "clean" records (no grounded defects but ungrounded defect comments)
#      are excluded — they were being trained as "find nothing" on diffs with real defects.
#   5. mid_eval now uses the same 12000-char budgeted prompts as the v4 baseline preds
#      and swaps checkpoints over merged-v4 (deltas are on merged-v4 now).
#   6. v5.2 reward constants: RECALL_BETA=1.5, CLEAN_CLAIM_PENALTY=0.35 (between v5.0's
#      over-quiet F1 and v5.1's over-loud beta=2).
#
# NOTE: checkpoints go to corpo-out-v5.2 — do NOT mix with corpo-out-v5: the old v5.0/v5.1
# checkpoints are LoRA deltas on base+v4-adapter, the new ones are deltas on merged-v4.

# Local copy of merged v4 first (loading 14 GB straight from Drive is slow, and Drive
# reads can stall mid-load). ~2-4 min once per session.
import os
if not os.path.exists('/content/sft-v4-merged/config.json'):
    !cp -r /content/drive/MyDrive/sft/sft-v4-merged-for-eval /content/sft-v4-merged
V4_MERGED = '/content/sft-v4-merged'
print(f"v4 merged (local): {V4_MERGED}")

!python /content/corpo_train.py \
    --v4-adapter {V4_ADAPTER} \
    --v4-merged {V4_MERGED} \
    --v4-backup {V4_BACKUP} \
    --train-prompts /content/ood_train_prompts.jsonl \
    --defect-labels /content/cache/defect_labels_train.jsonl \
    --output-dir /content/corpo-out \
    --checkpoint-sync-dir /content/drive/MyDrive/sft/corpo-out-v5.2 \
    --r-min-correct {R_MIN} \
    --kl-beta 0.02 \
    --learning-rate 5e-6 \
    --num-generations 8 \
    --prompts-per-step 4 \
    --max-new-tokens 2048 \
    --epochs 1 \
    --checkpoint-every 75 \
    --copy-to /content/drive/MyDrive/sft/code-reviewer-lora-v5.2-verifiable

# RESUME after a Colab disconnect: the local /content/corpo-out is wiped, but checkpoints
# are mirrored to Drive. Re-run Cells 0-1, re-run Cell 2 (labels) or restore the cache,
# then re-run THIS cell with --resume added, pointing at the latest Drive checkpoint, e.g.:
#   --resume /content/drive/MyDrive/sft/corpo-out-v5.2/checkpoint-150
# (batch note: per_device_train_batch_size=num_generations=8, gradient_accumulation_steps=
#  prompts_per_step=4 -> 4 prompts/optimizer-step; 8 % 8 == 0 satisfies TRL's divisibility rule.)

v4 merged (local): /content/sft-v4-merged
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
[corpo_train] verified v4 backup at /content/drive/MyDrive/sft/code-reviewer-lora-v4-traces-backup
[corpo_train] pre-flight: TRL exposes loss_type ✓
[corpo_train] pre-flight: max_prompt_length is configurable (set to 6144 in training) ✓
[corpo_train] pre-flight: batch ok (pdtbs=num_generations=8, grad_accum=prompts_per_step=4) ✓
[corpo_train] skipped 12 ambiguous label records (no grounded defects but ungrounded defect comments exist)
[corpo_train] dropped 12/1500 prompts with ambiguous/missing defect labels
[corpo_train] loaded 1488 prompts + defect labels for 1488 instances (670 with >=1 defect, 818 clean)
[corpo_train] loading merged v4 (/content/sft-v4-merged) + fresh LoRA via FastLanguageModel
INFO 06-10 23:51:15 [vllm_utils.py:690] Unsloth: Patching vLLM v1 graph capture
==((====))==  Unsloth 2026.6.2: 

In [12]:
  !rm -rf /content/drive/MyDrive/sft/sft-v4-merged-for-eval
  !cp -r /content/sft-v4-merged /content/drive/MyDrive/sft/sft-v4-merged-for-eval

In [13]:
# Cell 5 — mid-eval all v5.2 checkpoints vs the v4 bar (defect_recall / fp_rate / halluc)
#
# Runs as a SUBPROCESS: in-cell vLLM fails in Colab/Jupyter (vLLM v1 calls sys.stdout.fileno(),
# which a notebook stdout doesn't support). mid_eval.py LoRA-swaps each checkpoint over
# MERGED V4 (v5.2 checkpoints are deltas on merged-v4, NOT on the raw base), generates the
# fixed 50-prompt subset with the SAME 12000-char budgeted prompts the v4 preds used, and
# scores with the precision-aware judge-independent score_v5. Prints deltas vs the v4 bar.
!cd /content/sft && git pull --ff-only && cp /content/sft/*.py /content/
!python /content/mid_eval.py \
    --checkpoint-root /content/drive/MyDrive/sft/corpo-out-v5.2 \
    --v4-preds /content/drive/MyDrive/sft/ood_preds_v4.jsonl \
    --labels /content/cache/defect_labels_eval.jsonl \
    --v4-merged /content/sft-v4-merged \
    --n-samples 50
# Read the printed table: a checkpoint "beats v4" iff defect_recall > v4 AND fp_rate <= v4 AND halluc <= v4.
# n=50 is a DIRECTIONAL smoke test only (can't resolve <±0.10); the real call is Cell 7's
# full-632 paired bootstrap. Set BEST_CHECKPOINT to the winner (or /content/corpo-out/final
# after a full run) for Cell 6.

Already up to date.
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.9.1+cu128).
[mid-eval] v4 bar (n=50): defect_recall=0.115 fp_rate=0.618 halluc=0.036
[mid-eval] checkpoints: ['checkpoint-75', 'checkpoint-150', 'checkpoint-225', 'checkpoint-300', 'checkpoint-372']
INFO 06-11 06:56:44 [utils.py:261] non-default args: {'max_model_len': 8192, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'enable_lora': True, 'max_lora_rank': 64, 'model': '/content/sft-v4-merged'}
INFO 06-11 06:56:44 [model.py:541] Resolved architecture: Qwen2ForCausalLM
INFO 06-11 06:56:44 [model.py:1561] Using max model len 8192
INFO 06-11 06:56:44 [scheduler.py:226] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 06-11 06:56:44 [vllm.py:624] Asynchronous scheduling is enabled.
(EngineCore_DP0 pid=135800) INFO 06-11 06:56:45 [core.py:96] Initializing a V1 LLM engine (v0.15.1) with config: model='/content/sft-v4-merged', specu

In [14]:
# Cell 6 — Verify chat_template parity, merge BEST_CHECKPOINT onto MERGED V4, generate v5 preds
#
# Uses BEST_CHECKPOINT from Cell 5. Falls back to /content/corpo-out/final if Cell 5 was skipped.
# IMPORTANT (v5.2): checkpoints are LoRA deltas ON MERGED V4 — merging them onto the raw
# base would silently produce base+delta (the v4 weights would be missing).

# 6a. chat_template parity (else run_ood_eval.py's assert fires mid-run)
from transformers import AutoTokenizer
v4_tok = AutoTokenizer.from_pretrained(V4_ADAPTER)
base_tok = AutoTokenizer.from_pretrained('unsloth/Qwen2.5-Coder-7B-Instruct')
assert v4_tok.chat_template == base_tok.chat_template, \
    "v4 chat_template differs from base — patch run_ood_eval.py to load tokenizer per-model"
del v4_tok, base_tok
print("[cell6] chat_template parity: OK")

# 6b. which checkpoint
try:
    _src = BEST_CHECKPOINT
    print(f"[cell6] using BEST_CHECKPOINT from Cell 5: {_src}")
except NameError:
    _src = '/content/corpo-out/final'
    print(f"[cell6] Cell 5 skipped — falling back to {_src}")

# 6c. merge v5 adapter onto MERGED V4 (vLLM eval needs a full model)
import gc, os, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

if not os.path.exists('/content/sft-v4-merged/config.json'):
    !cp -r /content/drive/MyDrive/sft/sft-v4-merged-for-eval /content/sft-v4-merged

base = AutoModelForCausalLM.from_pretrained('/content/sft-v4-merged', dtype=torch.bfloat16)
merged = PeftModel.from_pretrained(base, _src).merge_and_unload()
merged.save_pretrained('/content/sft-v5-merged-for-eval', safe_serialization=True)  # local first, Drive later
AutoTokenizer.from_pretrained(V4_ADAPTER).save_pretrained('/content/sft-v5-merged-for-eval')
del merged, base
gc.collect(); torch.cuda.empty_cache()

# 6d. generate v5 predictions on the 632 OOD set (--skip-base: base preds already in ood_preds_v4.jsonl)
!python /content/run_ood_eval.py \
    --input /content/ood_input.jsonl \
    --output /content/ood_preds_v5.jsonl \
    --v4-model /content/sft-v5-merged-for-eval \
    --skip-base

AttributeError: 'list' object has no attribute 'keys'

In [16]:
BEST_CHECKPOINT = '/content/drive/MyDrive/sft/corpo-out-v5.2/checkpoint-150'

In [17]:
  # Cell 6 — merge BEST_CHECKPOINT onto MERGED V4, generate v5 preds (fixed: no adapter-tokenizer loads)
  import os, glob, json, shutil, gc, torch
  from transformers import AutoModelForCausalLM, AutoTokenizer
  from peft import PeftModel

  def _shards_ok(d):
      idx = os.path.join(d, 'model.safetensors.index.json')
      if not os.path.exists(idx): return False
      total = json.load(open(idx)).get('metadata', {}).get('total_size', 0)
      have = sum(os.path.getsize(p) for p in glob.glob(os.path.join(d, '*.safetensors')))
      return total > 0 and have >= total

  def _dir_chat_template(d):  # read template from files; adapter tokenizer crashes 4.56.2 if instantiated
      p = os.path.join(d, 'chat_template.jinja')
      if os.path.exists(p): return open(p).read()
      return json.load(open(os.path.join(d, 'tokenizer_config.json'))).get('chat_template')

  base_tok = AutoTokenizer.from_pretrained('unsloth/Qwen2.5-Coder-7B-Instruct')

  # 6a. chat_template parity
  _tpl = _dir_chat_template(V4_ADAPTER)
  assert _tpl is None or _tpl == base_tok.chat_template, "v4 chat_template differs from base!"
  print("[cell6] chat_template parity: OK")

  # 6b. checkpoint
  _src = BEST_CHECKPOINT
  print(f"[cell6] using BEST_CHECKPOINT: {_src}")

  # 6c. shard-verified merged v4, then merge the v5 checkpoint ONTO it (not onto raw base!)
  V4_MERGED = '/content/sft-v4-merged'
  assert _shards_ok(V4_MERGED), 'merged v4 missing/corrupt — re-run the repair cell first'

  base = AutoModelForCausalLM.from_pretrained(V4_MERGED, dtype=torch.bfloat16)
  merged = PeftModel.from_pretrained(base, _src).merge_and_unload()
  merged.save_pretrained('/content/sft-v5-merged-for-eval', safe_serialization=True)  # local first
  base_tok.save_pretrained('/content/sft-v5-merged-for-eval')
  del merged, base
  gc.collect(); torch.cuda.empty_cache()
  assert _shards_ok('/content/sft-v5-merged-for-eval'), 'v5 merge failed shard check'
  print("[cell6] v5 merged + shard-verified")

  # 6d. generate v5 predictions on the full 632 OOD set
  !python /content/run_ood_eval.py \
      --input /content/ood_input.jsonl \
      --output /content/ood_preds_v5.jsonl \
      --v4-model /content/sft-v5-merged-for-eval \
      --skip-base

[cell6] chat_template parity: OK
[cell6] using BEST_CHECKPOINT: /content/drive/MyDrive/sft/corpo-out-v5.2/checkpoint-150


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

[cell6] v5 merged + shard-verified
[run_ood_eval] 632 rows; generating v4 only (--skip-base)
[run_ood_eval] loaded tokenizer from unsloth/Qwen2.5-Coder-7B-Instruct
[run_ood_eval] chat_template matches v4_model: OK
[run_ood_eval] loading v4 (/content/sft-v5-merged-for-eval)
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.9.1+cu128).
INFO 06-11 07:24:44 [utils.py:261] non-default args: {'max_model_len': 8192, 'disable_log_stats': True, 'model': '/content/sft-v5-merged-for-eval'}
INFO 06-11 07:24:44 [model.py:541] Resolved architecture: Qwen2ForCausalLM
INFO 06-11 07:24:44 [model.py:1561] Using max model len 8192
INFO 06-11 07:24:44 [scheduler.py:226] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 06-11 07:24:44 [vllm.py:624] Asynchronous scheduling is enabled.
(EngineCore_DP0 pid=143579) INFO 06-11 07:24:45 [core.py:96] Initializing a V1 LLM engine (v0.15.1) with config: model='/content/sft-v5-merged-for-e

In [18]:
# Cell 7 — v5 vs v4 on the FULL 632 OOD set (judge-independent) — PAIRED-significance verdict
#
# Two judge-independent views:
#   (A) POINT estimates (defect_recall / fp_rate / halluc) — directly comparable to the goal bar.
#   (B) PAIRED bootstrap (compare_recall.py) — per-record delta v5-v4 with a 95% CI. THE TRUSTWORTHY CALL.
#       Comparing two MEANS (view A) misses a real +0.04 recall gain ~90% of the time at this n
#       (per-record recall std ~0.4 over ~150 labeled records → SE on the mean ~0.033 > the gain).
#       The paired test cancels per-record difficulty and detects the same gain ~100% of the time.
#
# SHIP v5 iff: recall is a SIGNIFICANT paired improvement (CI lower bound > 0)
#             AND fp_rate not significantly worse AND halluc not significantly worse.
import json, sys
sys.path.insert(0, '/content')
import score_v5, compare_recall

labels = {}
for l in open('/content/cache/defect_labels_eval.jsonl'):
    if l.strip():
        r = json.loads(l); labels[r['instance_id']] = r.get('defects', [])

def _load(p):
    return [json.loads(l) for l in open(p) if l.strip()]

v4_preds = _load('/content/drive/MyDrive/sft/ood_preds_v4.jsonl')
v5_preds = _load('/content/ood_preds_v5.jsonl')   # run_ood_eval wrote v5 output under 'v4_pred'

# Backfill diff into v5 records from v4 (same instance_id) so grounding/halluc is computed
# on the REAL diff even if run_ood_eval didn't echo the diff field.
_diff_by = {r['instance_id']: r.get('diff', '') for r in v4_preds}
for r in v5_preds:
    if not r.get('diff'):
        r['diff'] = _diff_by.get(r['instance_id'], '')

# (A) POINT estimates — the literal-bar view
print("scoring v4 (632, parallel)..."); v4 = score_v5.score(v4_preds, labels, 'v4_pred')
print("scoring v5 (632, parallel)..."); v5 = score_v5.score(v5_preds, labels, 'v4_pred')
def _dr(s): return s['defect_recall_labeled'] if s['defect_recall_labeled'] is not None else 0.0
def _fp(s): return s['fp_rate_clean'] if s['fp_rate_clean'] is not None else 1.0
print(f"\n[POINT]  {'metric':16s} {'v4':>8} {'v5':>8} {'delta':>9}")
for name, fn in [('defect_recall', _dr), ('fp_rate(clean)', _fp), ('halluc', lambda s: s['halluc_mean'])]:
    a, b = fn(v4), fn(v5); print(f"         {name:16s} {a:>8.3f} {b:>8.3f} {b-a:>+9.3f}")

# (B) PAIRED bootstrap — the trustworthy verdict (both files store the model output under 'v4_pred')
print("\n[PAIRED] bootstrapping per-record deltas (v5 - v4), n_boot=2000 ...")
pd = compare_recall.paired_delta(v4_preds, v5_preds, labels, 'v4_pred', 'v4_pred', n_boot=2000, seed=0)
def _ci(c): return f"[{c[0]:+.4f}, {c[1]:+.4f}]"
print(f"         compared {pd['n_compared']} (labeled {pd['n_labeled']}, clean {pd['n_clean']})")
print(f"         recall  delta={pd['recall_delta']:+.4f}  CI={_ci(pd['recall_ci'])}  "
      f"{'SIG IMPROVEMENT' if pd['recall_significant'] else 'not sig'}")
print(f"         fp      delta={pd['fp_delta']:+.4f}  CI={_ci(pd['fp_ci'])}  "
      f"{'sig better' if pd['fp_significant'] else 'not sig'}")
print(f"         halluc  delta={pd['halluc_delta']:+.4f}  CI={_ci(pd['halluc_ci'])}  "
      f"{'sig better' if pd['halluc_significant'] else 'not sig'}")

# fp/halluc "not significantly worse" = their delta CI is NOT entirely above 0
fp_not_worse = pd['fp_ci'][0] <= 0
halluc_not_worse = pd['halluc_ci'][0] <= 0
ship = pd['recall_significant'] and fp_not_worse and halluc_not_worse
print(f"\nVERDICT: {'SHIP v5 — significant paired recall gain; fp & halluc not significantly worse' if ship else 'KEEP v4 — recall gain not significant (or fp/halluc significantly worse)'}")
print("(Paired bootstrap is the call; the POINT table above is the literal-bar view. n=632, judge-independent.)")
json.dump({'point': {'v4': v4, 'v5': v5}, 'paired': pd, 'ship': ship},
          open('/content/v5_final_verdict.json', 'w'), indent=2)


scoring v4 (632, parallel)...
scoring v5 (632, parallel)...

[POINT]  metric                 v4       v5     delta
         defect_recall       0.087    0.093    +0.006
         fp_rate(clean)      0.749    0.658    -0.091
         halluc              0.036    0.029    -0.007

[PAIRED] bootstrapping per-record deltas (v5 - v4), n_boot=2000 ...
         compared 632 (labeled 281, clean 351)
         recall  delta=-0.0179  CI=[-0.0647, +0.0258]  not sig
         fp      delta=-0.1026  CI=[-0.1652, -0.0427]  sig better
         halluc  delta=-0.0074  CI=[-0.0233, +0.0075]  not sig

VERDICT: KEEP v4 — recall gain not significant (or fp/halluc significantly worse)
(Paired bootstrap is the call; the POINT table above is the literal-bar view. n=632, judge-independent.)


In [19]:
  !cp /content/ood_preds_v5.jsonl /content/v5_final_verdict.json /content/drive/MyDrive/sft/

In [20]:

  import json, re, random, statistics

  v4 = {r['instance_id']: r for r in map(json.loads, open('/content/drive/MyDrive/sft/ood_preds_v4.jsonl'))}
  v5 = {r['instance_id']: r for r in map(json.loads, open('/content/ood_preds_v5.jsonl'))}
  labels = {r['instance_id']: r.get('defects', []) for r in map(json.loads, open('/content/cache/defect_labels_eval.jsonl'))}
  ids = [i for i in v4 if i in v5]

  def stats(get, name):
      lens, idents, clean, loops = [], [], 0, 0
      for i in ids:
          t = get(i) or ''
          lens.append(len(t)); idents.append(len(re.findall(r"`([A-Za-z_][A-Za-z0-9_]{1,40})`", t)))
          if any(k in t.lower() for k in ('no issue','looks correct','looks good','no bug','no significant','no major','lgtm')): clean += 1
          if re.search(r'(.{15,80}?)\1{9,}', t, re.DOTALL): loops += 1
      print(f"{name:5s} mean_len={statistics.mean(lens):6.0f} median={statistics.median(lens):5.0f} "
            f"idents/rev={statistics.mean(idents):4.1f} clean-verdict={clean/len(ids):.3f} loops={loops}")

  stats(lambda i: v4[i]['v4_pred'], 'v4'); stats(lambda i: v5[i]['v4_pred'], 'v5.2')

  ASSERT = re.compile(r'\b(bug|issue|error|incorrect|wrong|missing|fail|broken|vulnerab|leak|race)\b', re.I)
  QUIET = re.compile(r'(no (major |significant |obvious |apparent )?(issues?|bugs?|problems?)|looks (good|correct|fine)|lgtm)', re.I)
  rng = random.Random(0)

  cands = [i for i in ids if labels.get(i) == [] and ASSERT.search(v4[i]['v4_pred'] or '')
           and (QUIET.search(v5[i]['v4_pred'] or '') or not ASSERT.search(v5[i]['v4_pred'] or ''))]
  print(f"\n=== A) clean diffs where v4 flags but v5.2 stays quiet ({len(cands)} total, 8 shown) ===")
  for i in rng.sample(cands, min(8, len(cands))):
      print(f"\n--- {i} ({v4[i].get('repo')}) ---")
      print("DIFF:", v4[i].get('diff','')[:450].replace('\n',' ⏎ '))
      print("V4  :", (v4[i]['v4_pred'] or '')[:350].replace('\n',' '))
      print("V5.2:", (v5[i]['v4_pred'] or '')[:350].replace('\n',' '))

  print("\n=== B) labeled records with ground truth (6 random) ===")
  for i in rng.sample([i for i in ids if labels.get(i)], 6):
      print(f"\n--- {i} ---")
      for d in labels[i][:3]:
          print("GT  :", d.get('path'), d.get('line'), d.get('issue_type'), '—', (d.get('canonical_desc') or '')[:150])
      print("V4  :", (v4[i]['v4_pred'] or '')[:330].replace('\n',' '))
      print("V5.2:", (v5[i]['v4_pred'] or '')[:330].replace('\n',' '))

v4    mean_len=   898 median=  275 idents/rev= 6.0 clean-verdict=0.258 loops=0
v5.2  mean_len=   516 median=  261 idents/rev= 3.2 clean-verdict=0.334 loops=4

=== A) clean diffs where v4 flags but v5.2 stays quiet (59 total, 8 shown) ===

--- py-pdf__pypdf-1835@5a9f0ab (py-pdf/pypdf) ---
DIFF: diff --git a/docs/user/file-size.md b/docs/user/file-size.md ⏎ index a7b2d3cc4..7748ce2aa 100644 ⏎ --- a/docs/user/file-size.md ⏎ +++ b/docs/user/file-size.md ⏎ @@ -30,7 +30,7 @@ It depends on the PDF how well this works, but we have seen an 86% file ⏎  reduction (from 5.7 MB to 0.8 MB) within a real PDF. ⏎   ⏎   ⏎ -## Remove images ⏎ +## Removing Images ⏎   ⏎   ⏎  ```python ⏎ @@ -75,3 +75,10 @@ with open("out.pdf", "wb") as f: ⏎   ⏎  Using this method, we have seen a reducti
V4  : Missing space after “persist” in “will persist(the data…” and “deleting” → “deleted”. “Cropping / (adjust viewboxes)” is awkward; consider “Cropping (adjust viewboxes)”. “Making dropping refrences useless” is garbled—l

In [21]:

  import json, re
  from collections import Counter

  v5 = {r['instance_id']: r for r in map(json.loads, open('/content/ood_preds_v5.jsonl'))}

  print("=== the 4 loop records ===")
  for i, r in v5.items():
      t = r['v4_pred'] or ''
      m = re.search(r'(.{15,80}?)\1{9,}', t, re.DOTALL)
      if m:
          print(f"\n--- {i} ({r.get('repo')}) len={len(t)}")
          print("repeating unit:", repr(m.group(1))[:120])
          print("head:", t[:200].replace('\n',' '))

  print("\n=== eval label composition by issue_type ===")
  c, recs = Counter(), 0
  for line in open('/content/cache/defect_labels_eval.jsonl'):
      r = json.loads(line)
      if r.get('defects'): recs += 1
      for d in r.get('defects', []): c[d.get('issue_type')] += 1
  print(f"labeled records: {recs}")
  for k, v in c.most_common(): print(f"  {k:15s} {v}")


=== the 4 loop records ===

--- apache__airflow-51237@6dd1634 (apache/airflow) len=15775
repeating unit: '...\n...\n...\n...\n'
head: <think> We have a diff that shows changes to an SVG file. The diff is:  ``` @@ -1,4 +1,4 @@  ... -<svg class="rich-terminal" viewBox="0 0 1482 1099.2" xmlns="http://www.w3.org/2000/svg"> +<svg class="

--- getsentry__sentry-73239@4060eaa (getsentry/sentry) len=15659
repeating unit: '\n-  const {rootProps, setTabListState} = useContext(TabsContext);\n...'
head: <think> We need to review the provided diff. The diff shows changes to two files: `static/app/components/draggableTabs/draggableTab.tsx` and `static/app/components/draggableTabs/draggableTabList.tsx`.

--- skypilot-org__skypilot-1622@2c08184 (skypilot-org/skypilot) len=15094
repeating unit: ',\n    ...\n    cpus: Optional[str]'
head: <think> We need to review the provided diff. The diff shows modifications in `sky/cli.py` and `sky/clouds/aws.py`. Let's analyze.  First, the diff for `sky/cli.py`:  -

In [22]:
  !cp -r /content/drive/MyDrive/sft/corpo-out-v5.2/checkpoint-150 /content/drive/MyDrive/sft/code-reviewer-lora-v5.2-ckpt150-best